# 08 消融实验与综合评价

## 本课学习目标

- A. 量化金融主线：消融实验（Ablation Study）
零基础解释：一次只加入或去掉一个模块，观察它是否真的带来改进。
- B. 大语言模型主线：检索增强生成（Retrieval-Augmented Generation，RAG）和 TF-IDF
零基础解释：本课只演示先检索再使用资料的思想，不实现真实在线 RAG。
- C. 两条线如何连接：把市场数据和新闻文本转成可检查的表格信号。
- D. 可运行实验：比较四组策略，输出净值曲线、指标表、混淆矩阵和检索结果。
- E. 结果解释：观察表格、图表和结构化输出。
- F. 常见错误：把回测收益当成未来收益、把 Mock 当成真实模型。
- G. 课后练习：修改一个参数并重新运行。
- H. 本课术语表：见本课各小节。

## 本课最终输出

一个离线实验输出，不联网、不调用真实模型、不产生真实订单。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "learning").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "learning" / "data"


## 从关键词检索到向量检索

### 嵌入向量（Embedding）

零基础解释：嵌入向量是将文字、图片或其他信息转换成一组数字，使计算机能够比较不同内容在语义上的相似程度。

关键理解：
- Embedding 通常是一个数值向量（例如 768 维或 1536 维的浮点数列表）；
- 意思相近的文本，其向量通常在空间中距离更近；
- Embedding 不是大语言模型的最终回答——它只是一个"表示层"；
- Embedding 常用于搜索、推荐、聚类和检索增强生成（RAG）；
- **本课程不下载真实 Embedding 模型**，只演示 TF-IDF 离线检索。

### 词频-逆文档频率（Term Frequency–Inverse Document Frequency，TF-IDF）

零基础解释：TF-IDF 根据一个词在当前文档中出现得是否频繁，以及它在全部文档中是否稀有，为这个词计算权重。

**词频（Term Frequency，TF）：**
- 某个词在当前文档中出现的次数 ÷ 文档总词数；
- 出现越频繁，TF 值越高。

**逆文档频率（Inverse Document Frequency，IDF）：**
- log(总文档数 ÷ 包含该词的文档数)；
- 该词在越多文档中出现，IDF 值越低；
- 常见词（如"的""是"）的 IDF 很低。

**TF-IDF 的计算：**
```
TF-IDF = TF × IDF
```
- 在当前文档中高频、但在全部文档中稀有的词，TF-IDF 权重最高；
- 常见词（几乎所有文档都出现）权重较低；
- 具有区分度的词（只在少数文档出现）权重较高。

**TF-IDF 的局限性：**
- TF-IDF 主要比较词语的重合度（字面匹配）；
- TF-IDF ≠ 语义 Embedding：两个意思相近但用词不同的句子，TF-IDF 可能认为相似度很低；
- TF-IDF 不能真正"理解"语言——它只是统计词频。

### 三种检索方法对比

| 方法 | 是否需要模型 | 是否理解语义 | 本课程是否实际运行 |
|------|:----------:|:----------:|:----------------:|
| 关键词匹配 | 否 | 否 | 是 |
| TF-IDF | 否 | 很弱 | 是 |
| Embedding | 通常需要 | 较强 | 否（仅演示概念）|
| 大语言模型 | 是 | 较强 | Mock 演示 |

### 检索增强生成（Retrieval-Augmented Generation，RAG）

RAG 的完整流程：

```
用户问题
    ↓
① 检索相关资料（本课用 TF-IDF 查询 sample_company_notes.csv）
    ↓
② 将检索到的资料与用户问题拼接
    ↓
③ 一起交给大语言模型
    ↓
④ 模型基于资料回答（而非仅凭训练记忆）
```

重要提醒：
- RAG 不能保证完全没有幻觉——检索结果可能错误或不完整；
- 检索到的资料可能不相关或被误解；
- 必须保留资料来源，以便追溯和验证；
- **本阶段只演示离线检索（步骤 ①），不调用真实模型完成步骤 ③④**。

下面代码同时运行消融实验的四种策略，并演示 TF-IDF 离线检索。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from learning.src.plotting import configure_chinese_plotting, safe_title
from learning.src.market_data import load_price_data
from learning.src.momentum_factor import compute_momentum, rank_momentum
from learning.src.sentiment_factor import classify_news, daily_sentiment_factor
from learning.src.time_alignment import monthly_signal_schedule
from learning.src.mini_backtest import run_backtest
from learning.src.financial_metrics import simple_returns, cumulative_return, maximum_drawdown, sharpe_ratio
from learning.src.retrieval_demo import retrieve_notes

# 配置中文字体
_font = configure_chinese_plotting()

prices = load_price_data(DATA / "sample_prices.csv")
news_raw = pd.read_csv(DATA / "sample_news.csv")
news = classify_news(news_raw)
schedule = monthly_signal_schedule(prices).head(8)
mom = compute_momentum(prices, 20)

def make_targets(mode):
    rows = []
    for _, s in schedule.iterrows():
        base = rank_momentum(mom, s["signal_date"], 20)
        sent = daily_sentiment_factor(news, s["signal_timestamp"]).groupby("ticker")["sentiment_score"].mean()
        base["sentiment_score"] = base["ticker"].map(sent).fillna(0)
        if mode == "buy_hold":
            picks = ["AAA", "BBB"]
        elif mode == "momentum":
            picks = base.nlargest(2, "momentum_rank_score")["ticker"].tolist()
        elif mode == "sentiment":
            picks = base.nlargest(2, "sentiment_score")["ticker"].tolist()
        else:
            base["combined"] = 0.7 * base["momentum_rank_score"] + 0.3 * ((base["sentiment_score"] + 1) / 2)
            picks = base.nlargest(2, "combined")["ticker"].tolist()
        for ticker in picks:
            rows.append({"execution_date": s["execution_date"], "ticker": ticker, "weight": 0.5})
    return pd.DataFrame(rows)

curves = {}
rows = []
for mode in ["buy_hold", "momentum", "sentiment", "combined"]:
    result = run_backtest(prices, make_targets(mode))
    eq = result.equity_curve.set_index("date")["equity"]
    curves[mode] = eq / eq.iloc[0]
    rets = simple_returns(eq)
    rows.append({"mode": mode, "cumulative": cumulative_return(rets), "max_drawdown": maximum_drawdown(eq), "sharpe": sharpe_ratio(rets)})

fig, ax = plt.subplots(figsize=(10, 5))
pd.DataFrame(curves).plot(ax=ax)
safe_title(ax, "消融实验：四种策略净值对比（合成教学数据）", "Ablation Study: Four Strategy NAVs (synthetic data)")
ax.set_xlabel("日期")
ax.set_ylabel("净值（初始=1）")
ax.legend(["买入持有", "仅动量", "仅情绪", "联合策略"])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(pd.DataFrame(rows))
display(pd.DataFrame(confusion_matrix(news_raw["expected_label"], news["label"], labels=["positive","neutral","negative"]), index=["expected_positive","expected_neutral","expected_negative"], columns=["pred_positive","pred_neutral","pred_negative"]))

print("=" * 50)
print("TF-IDF 离线检索演示：")
print("查询词：'model risk and teaching data'")
print("检索结果（前3条）：")
display(retrieve_notes(DATA / "sample_company_notes.csv", "model risk and teaching data", 3))
print("结果仅用于教学，不代表未来收益。")
print("TF-IDF 基于词频统计，不涉及语义理解或 Embedding 模型。")
print("=" * 50)

## 结尾总结

本课通过消融实验系统比较了买入持有、仅动量、仅情绪和动量情绪联合四种策略。

本课核心收获：
- 消融实验通过逐一移除模块，判断每个模块是否真正贡献了价值；
- TF-IDF 是基于词频统计的检索方法，不涉及语义理解；
- Embedding 向量可以捕捉语义相似度，但本课程不下载真实模型；
- RAG（检索增强生成）的流程：检索 → 拼接 → 提问 → 基于资料回答；
- 混淆矩阵可以评估 Mock Provider 的情绪分类准确性。

哪些结果不能解释为策略一定赚钱：任何图表和收益数字都只是合成数据上的教学结果。

本课使用了哪些英文专业词：
- Ablation Study（消融实验）
- Term Frequency-Inverse Document Frequency（TF-IDF，词频-逆文档频率）
- Embedding（嵌入向量）
- Retrieval-Augmented Generation（RAG，检索增强生成）
- Confusion Matrix（混淆矩阵）
- Buy and Hold（买入并持有）

### 常见错误

1. **不跑消融就直接下结论**：联合策略表现好，不一定是因为"联合"有效——可能动量因子本身就已经足够。
2. **把 TF-IDF 当成语义理解**：TF-IDF 只看词语重合，两个意思相同但用词不同的句子会被误判为不相似。
3. **认为 RAG 能完全消除幻觉**：RAG 能减少幻觉，但检索到的资料也可能是错误的或不完整的。
4. **混淆 Embedding 和 TF-IDF**：Embedding 是稠密向量、捕捉语义；TF-IDF 是稀疏向量、基于词频。

### 课后练习

1. **去掉情绪因子**：将联合策略中情绪权重设为 0（只用动量），重新运行消融实验，观察情绪模块是否提供了增量价值。
2. **修改检索查询**：换一个查询词（如 "business" 或 "risk"），观察 TF-IDF 检索结果是否发生变化。
3. **解释混淆矩阵**：观察混淆矩阵中哪个类别被误判最多，分析可能的原因（提示：查看 MockLLMProvider 的关键词匹配逻辑）。

下一课与本课有什么关系：Phase 1 的八课学习到此结束。你现在已经具备了量化金融和大语言模型的基础知识，可以继续学习 Phase 2 或探索原项目代码。